In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table("bronze.world_bank_food_prices")

print("Bronze rows:", bronze_df.count())

In [0]:
required_columns = [
    "ISO3",
    "country",
    "geo_id",
    "price_date",
    "lat",
    "lon"
]

for column in required_columns:
    if column in bronze_df.columns:
        print(column, "exists")
    else:
        print(column, "MISSING")

In [0]:
validated_df = (
    bronze_df
    .withColumn("_price_date", F.to_date(F.col("price_date")))
    .withColumn("_lat", F.col("lat").cast("double"))
    .withColumn("_lon", F.col("lon").cast("double"))
)

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "_validation_error",
        F.when(
            F.trim(F.col("ISO3")) == "",
            F.lit("Missing ISO3")
        )
        .when(
            F.trim(F.col("country")) == "",
            F.lit("Missing country")
        )
        .when(
            F.trim(F.col("geo_id")) == "",
            F.lit("Missing geo_id")
        )
        .when(
            F.col("_price_date").isNull(),
            F.lit("Invalid price_date")
        )
        .when(
            F.col("_lat").isNull(),
            F.lit("Invalid latitude")
        )
        .when(
            F.col("_lon").isNull(),
            F.lit("Invalid longitude")
        )
    )
)

In [0]:
valid_df = validated_df.filter(
    F.col("_validation_error").isNull()
)

reject_df = validated_df.filter(
    F.col("_validation_error").isNotNull()
)

print("Valid records:", valid_df.count())
print("Rejected records:", reject_df.count())

In [0]:
reject_table = "bronze.world_bank_food_price_rejects"

(
    reject_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(reject_table)
)

print("Reject table created:", reject_table)

In [0]:
display(
    spark.sql("""
        select count(*) as rejected_records
        from bronze.world_bank_food_price_rejects
    """)
)

In [0]:
spark.sql("create schema if not exists audit")

In [0]:
spark.sql("""
create table if not exists audit.pipeline_runs (
    run_id string,
    source_name string,
    layer string,
    load_type string,
    start_time timestamp,
    end_time timestamp,
    records_read long,
    records_inserted long,
    records_updated long,
    records_rejected long,
    status string,
    error_message string
)
using delta
""")

In [0]:
validation_run_id = "a4f57005-016f-4a68-9518-f1aa342dc9f6"

spark.sql(f"""
insert into audit.pipeline_runs
values (
    '{validation_run_id}',
    'World Bank Real-Time Food Prices',
    'Bronze',
    'full',
    current_timestamp(),
    current_timestamp(),
    {bronze_df.count()},
    {valid_df.count()},
    0,
    {reject_df.count()},
    'SUCCESS',
    null
)
""")

In [0]:
display(
    spark.sql("""
        select *
        from audit.pipeline_runs
        order by start_time desc
    """)
)

In [0]:
spark.sql("create schema if not exists silver")

In [0]:
silver_df = (
    valid_df
    .withColumn("price_date", F.to_date(F.col("price_date")))
    .withColumn("year", F.expr("try_cast(year as int)"))
    .withColumn("month", F.expr("try_cast(month as int)"))
    .withColumn("lat", F.expr("try_cast(lat as double)"))
    .withColumn("lon", F.expr("try_cast(lon as double)"))
)

In [0]:
validation_columns = [
    "_price_date",
    "_lat",
    "_lon",
    "_validation_error"
]

for column in validation_columns:
    if column in silver_df.columns:
        silver_df = silver_df.drop(column)

In [0]:
silver_df = silver_df.dropDuplicates(
    ["ISO3", "geo_id", "price_date"]
)

In [0]:
display(
    silver_df.select(
        "ISO3",
        "country",
        "geo_id",
        "price_date",
        "year",
        "month",
        "lat",
        "lon"
    ).limit(20)
)

In [0]:
#silver row count
display(
    silver_df.agg(
        F.count("*").alias("silver_rows")
    )
)

In [0]:
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.world_bank_food_prices")
)

print("Silver table created successfully")

In [0]:
spark.sql("""
    describe silver.world_bank_food_prices
""").show(100, truncate=False)

In [0]:
display(
    spark.sql("""
        select
            min(price_date) as earliest_date,
            max(price_date) as latest_date,
            count(*) as total_rows
        from silver.world_bank_food_prices
    """)
)